In [1]:
# ============================================
# HUP seizure preprocessing: bipolar_0p5_60_128_fixed128
# ============================================

from __future__ import annotations

import json
import math
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import mne
import numpy as np
import pandas as pd


C:\Users\ajars\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# ============================================================
# CONFIG
# ============================================================
CONFIG = {
    # paths
    "RAW_ROOT": Path(r"D:\HUP dataset"),
    "OUT_ROOT": Path(r"D:\HUP_processed_ver2"),

    # filters
    "SUBJECT_FILTER": [],          # e.g. ["sub-HUP146"]
    "TASK_FILTER": [],             # leave empty to process both ictal + interictal
    "ACQ_FILTER": [],              # e.g. ["seeg", "ecog"]

    # preprocessing
    "REFERENCE_MODE": "bipolar",   # "bipolar" or "car"
    "NOTCH_FREQS": [60.0, 120.0],
    "L_FREQ": 0.5,
    "H_FREQ": 60.0,
    "RESAMPLE_HZ": 128.0,
    "NORMALIZE_MODE": "robust_zscore_per_channel",

    # windowing
    "WINDOW_LEN_SEC": 3.0,
    "TRAIN_HOP_ICTAL_SEC": 0.5,
    "TRAIN_HOP_NONICTAL_SEC": 1.0,
    "TEST_HOP_SEC": 1.0,

    # labeling
    "MIN_ICTAL_OVERLAP_FRAC": 0.50,
    "AMBIG_LOWER": 0.10,
    "AMBIG_UPPER": 0.50,

    # channel handling
    "TARGET_N_CHANNELS": 128,
    "RANDOM_SEED": 42,

    # artifact summaries only; not used for hard rejection by default
    "ARTIFACT_RMS_ZTHRESH": 7.0,
    "ARTIFACT_PEAK_ZTHRESH": 7.0,

    # saving
    "OVERWRITE": True,
}

In [3]:
# ============================================================
# DATA CLASSES
# ============================================================
@dataclass
class PipelineConfig:
    name: str
    reference: str
    band: Tuple[float, float]
    resample: float
    normalize: str
    target_n_channels: int


@dataclass
class WindowingConfig:
    window_len_sec: float
    train_hop_nonictal_sec: float
    train_hop_ictal_sec: float
    test_hop_sec: float
    min_ictal_overlap_frac: float
    ambig_lower: float
    ambig_upper: float
    y_train_minus_one_means_ignore: bool = True


@dataclass
class RunMeta:
    pipeline: Dict
    subject: str
    session: str
    task: str
    acquisition: str
    run: str
    edf_path: str
    channels_tsv: str
    events_tsv: str
    json_path: str
    recording_duration_sec: float
    sfreq_before_resample: float
    sfreq_after_resample: float
    powerline_frequency_hz: Optional[float]
    ictal_intervals: List[Tuple[float, float]]
    windowing: Dict
    n_channels_before_drop: int
    n_channels_after_drop: int
    n_channels_after_reference: int
    n_channels_after_fix128: int
    kept_channel_indices: List[int]
    channel_names_after_reference: List[str]
    separability: Dict[str, float]
    class_counts: Dict[str, int]


PIPELINE = PipelineConfig(
    name="pipeline_c_bipolar_0p5_60_128_fixed128",
    reference=CONFIG["REFERENCE_MODE"],
    band=(CONFIG["L_FREQ"], CONFIG["H_FREQ"]),
    resample=CONFIG["RESAMPLE_HZ"],
    normalize=CONFIG["NORMALIZE_MODE"],
    target_n_channels=CONFIG["TARGET_N_CHANNELS"],
)

WINDOWING = WindowingConfig(
    window_len_sec=CONFIG["WINDOW_LEN_SEC"],
    train_hop_nonictal_sec=CONFIG["TRAIN_HOP_NONICTAL_SEC"],
    train_hop_ictal_sec=CONFIG["TRAIN_HOP_ICTAL_SEC"],
    test_hop_sec=CONFIG["TEST_HOP_SEC"],
    min_ictal_overlap_frac=CONFIG["MIN_ICTAL_OVERLAP_FRAC"],
    ambig_lower=CONFIG["AMBIG_LOWER"],
    ambig_upper=CONFIG["AMBIG_UPPER"],
)


In [4]:
# ============================================================
# HELPERS
# ============================================================
def normalize_contact_name(name: str) -> str:
    name = str(name).strip().upper()
    name = name.replace("-REF", "").replace("_REF", "").replace("REF", "")
    name = name.replace("EEG", "")
    name = re.sub(r"[^A-Z0-9]", "", name)
    return name


def split_contact_name(name: str) -> Tuple[Optional[str], Optional[int]]:
    name = normalize_contact_name(name)
    m = re.match(r"(.+?)(\d+)$", name)
    if not m:
        return None, None
    return m.group(1), int(m.group(2))


def robust_zscore_per_channel(x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    med = np.median(x, axis=1, keepdims=True)
    mad = np.median(np.abs(x - med), axis=1, keepdims=True)
    scale = 1.4826 * mad
    scale = np.where(scale < eps, 1.0, scale)
    return (x - med) / scale


def robust_window_summary(vec: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    med = np.median(vec)
    mad = np.median(np.abs(vec - med))
    scale = 1.4826 * mad
    if scale < eps:
        scale = 1.0
    return (vec - med) / scale


def read_json(path: Path) -> Dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None


def overlap_frac(a0: float, a1: float, b0: float, b1: float) -> float:
    inter = max(0.0, min(a1, b1) - max(a0, b0))
    denom = max(1e-9, a1 - a0)
    return inter / denom


def find_runs(raw_root: Path) -> List[Dict]:
    runs = []
    for edf_path in raw_root.glob("sub-*/ses-*/ieeg/*_ieeg.edf"):
        stem = edf_path.name.replace("_ieeg.edf", "")
        channels_tsv = edf_path.with_name(stem + "_channels.tsv")
        events_tsv = edf_path.with_name(stem + "_events.tsv")
        json_path = edf_path.with_name(stem + "_ieeg.json")

        if not channels_tsv.exists() or not json_path.exists():
            continue

        # events.tsv is optional now; interictal may have no seizure events
        if not events_tsv.exists():
            events_tsv = None

        m = re.match(
            r"(?P<sub>sub-[^_]+)_(?P<ses>ses-[^_]+)_task-(?P<task>[^_]+)_acq-(?P<acq>[^_]+)_run-(?P<run>[^_]+)",
            stem
        )
        if not m:
            continue

        runs.append({
            "subject": m.group("sub"),
            "session": m.group("ses"),
            "task": m.group("task"),
            "acquisition": m.group("acq"),
            "run": m.group("run"),
            "edf_path": edf_path,
            "channels_tsv": channels_tsv,
            "events_tsv": events_tsv,
            "json_path": json_path,
            "stem": stem,
        })
    return sorted(runs, key=lambda x: (x["subject"], x["task"], x["run"]))


def load_ictal_intervals(events_tsv: Optional[Path]) -> List[Tuple[float, float]]:
    if events_tsv is None or not Path(events_tsv).exists():
        return []

    df = pd.read_csv(events_tsv, sep="\t")
    if "trial_type" not in df.columns or "onset" not in df.columns:
        return []

    df = df.copy()
    df["trial_type"] = df["trial_type"].astype(str).str.strip().str.lower()
    df["onset"] = pd.to_numeric(df["onset"], errors="coerce")

    onsets = df[df["trial_type"] == "sz onset"]["onset"].dropna().tolist()
    offsets = df[df["trial_type"] == "sz offset"]["onset"].dropna().tolist()

    intervals = []
    for a, b in zip(onsets, offsets):
        if b > a:
            intervals.append((float(a), float(b)))
    return intervals


def choose_ieeg_channels(raw: mne.io.BaseRaw, channels_tsv: Path) -> mne.io.BaseRaw:
    cdf = pd.read_csv(channels_tsv, sep="\t")

    keep_types = {"ecog", "seeg"}
    if "type" in cdf.columns:
        cdf["type"] = cdf["type"].astype(str).str.lower().str.strip()
    else:
        cdf["type"] = ""

    if "status" in cdf.columns:
        cdf["status"] = cdf["status"].astype(str).str.lower().str.strip()
    else:
        cdf["status"] = "good"

    good_names = cdf[
        cdf["type"].isin(keep_types) &
        (~cdf["status"].isin({"bad"}))
    ]["name"].astype(str).tolist()

    good_names = [ch for ch in good_names if ch in raw.ch_names]
    if not good_names:
        raise RuntimeError(f"No usable iEEG channels found in {channels_tsv.name}")

    return raw.copy().pick(good_names)


def make_bipolar_adjacent(raw: mne.io.BaseRaw) -> mne.io.BaseRaw:
    groups: Dict[str, List[Tuple[int, str]]] = {}
    for ch in raw.ch_names:
        prefix, idx = split_contact_name(ch)
        if prefix is None:
            continue
        groups.setdefault(prefix, []).append((idx, ch))

    anodes, cathodes, new_names = [], [], []
    for prefix, items in groups.items():
        items = sorted(items, key=lambda x: x[0])
        for (i1, ch1), (i2, ch2) in zip(items[:-1], items[1:]):
            if i2 == i1 + 1:
                cathodes.append(ch1)
                anodes.append(ch2)
                new_names.append(f"{prefix}{i1}-{prefix}{i2}")

    if not anodes:
        raise RuntimeError("No adjacent bipolar pairs could be formed.")

    raw_bip = mne.set_bipolar_reference(
        raw,
        anode=anodes,
        cathode=cathodes,
        ch_name=new_names,
        drop_refs=False,
        copy=True,
    )
    raw_bip.pick(new_names)
    return raw_bip


def pad_or_trim_channels(X: np.ndarray, target_n_channels: int, seed: int = 42):
    rng = np.random.default_rng(seed)
    n_windows, n_channels, n_samples = X.shape

    if n_channels == target_n_channels:
        return X, np.ones(target_n_channels, dtype=bool), np.arange(n_channels)

    if n_channels < target_n_channels:
        X_fixed = np.zeros((n_windows, target_n_channels, n_samples), dtype=X.dtype)
        X_fixed[:, :n_channels, :] = X
        mask = np.zeros(target_n_channels, dtype=bool)
        mask[:n_channels] = True
        return X_fixed, mask, np.arange(n_channels)

    kept_idx = np.sort(rng.choice(n_channels, size=target_n_channels, replace=False))
    X_fixed = X[:, kept_idx]
    mask = np.ones(target_n_channels, dtype=bool)
    return X_fixed, mask, kept_idx


def separability_score(values: np.ndarray, y: np.ndarray) -> float:
    non = values[y == 0]
    ict = values[y == 1]
    if len(non) < 2 or len(ict) < 2:
        return float("nan")
    pooled = math.sqrt((non.var() + ict.var()) / 2.0)
    if pooled == 0:
        return 0.0
    return float((ict.mean() - non.mean()) / pooled)


In [5]:
# ============================================================
# WINDOWING
# ============================================================
def label_window(
    t0: float,
    t1: float,
    ictal_intervals: List[Tuple[float, float]],
    min_ictal_overlap_frac: float,
    ambig_lower: float,
    ambig_upper: float,
) -> Tuple[int, int]:
    max_frac = 0.0
    for s, e in ictal_intervals:
        max_frac = max(max_frac, overlap_frac(t0, t1, s, e))

    y = 1 if max_frac >= min_ictal_overlap_frac else 0

    if ambig_lower <= max_frac < ambig_upper:
        y_train = -1
    else:
        y_train = y
    return y, y_train


def build_window_starts(
    duration_sec: float,
    ictal_intervals: List[Tuple[float, float]],
    window_len_sec: float,
    hop_nonictal_sec: float,
    hop_ictal_sec: float,
) -> np.ndarray:
    starts = set()

    # base grid for all runs
    s = 0.0
    while s + window_len_sec <= duration_sec + 1e-9:
        starts.add(round(s, 6))
        s += hop_nonictal_sec

    # denser ictal grid only if ictal intervals exist
    for ict0, ict1 in ictal_intervals:
        s = max(0.0, ict0 - window_len_sec)
        end = min(duration_sec - window_len_sec, ict1)
        while s <= end + 1e-9:
            starts.add(round(s, 6))
            s += hop_ictal_sec

    return np.array(sorted(starts), dtype=np.float32)


def make_windows(
    data_ch_time: np.ndarray,
    sfreq: float,
    ictal_intervals: List[Tuple[float, float]],
    cfg: WindowingConfig,
) -> Dict[str, np.ndarray]:
    n_channels, n_samples = data_ch_time.shape
    duration_sec = n_samples / sfreq
    win_len_samp = int(round(cfg.window_len_sec * sfreq))

    starts_sec = build_window_starts(
        duration_sec=duration_sec,
        ictal_intervals=ictal_intervals,
        window_len_sec=cfg.window_len_sec,
        hop_nonictal_sec=cfg.train_hop_nonictal_sec,
        hop_ictal_sec=cfg.train_hop_ictal_sec,
    )

    X_list, y_list, yt_list, tb_list = [], [], [], []
    rms_raw, peak_raw = [], []

    for t0 in starts_sec:
        t1 = t0 + cfg.window_len_sec
        s0 = int(round(t0 * sfreq))
        s1 = s0 + win_len_samp
        if s1 > n_samples:
            continue

        window = data_ch_time[:, s0:s1]
        if window.shape[1] != win_len_samp:
            continue

        y, y_train = label_window(
            t0=float(t0),
            t1=float(t1),
            ictal_intervals=ictal_intervals,
            min_ictal_overlap_frac=cfg.min_ictal_overlap_frac,
            ambig_lower=cfg.ambig_lower,
            ambig_upper=cfg.ambig_upper,
        )

        rms_per_ch = np.sqrt(np.mean(window ** 2, axis=1))
        peak_per_ch = np.max(np.abs(window), axis=1)

        rms_raw.append(float(np.median(rms_per_ch)))
        peak_raw.append(float(np.median(peak_per_ch)))

        X_list.append(window.astype(np.float32))
        y_list.append(y)
        yt_list.append(y_train)
        tb_list.append([float(t0), float(t1)])

    X = np.stack(X_list, axis=0)
    y = np.asarray(y_list, dtype=np.int64)
    y_train = np.asarray(yt_list, dtype=np.int64)
    t_bounds = np.asarray(tb_list, dtype=np.float32)

    rms_z = robust_window_summary(np.asarray(rms_raw, dtype=np.float32)).astype(np.float32)
    peak_z = robust_window_summary(np.asarray(peak_raw, dtype=np.float32)).astype(np.float32)

    return {
        "X": X,
        "y": y,
        "y_train": y_train,
        "t_bounds": t_bounds,
        "rms_z": rms_z,
        "peak_z": peak_z,
    }


In [6]:
# ============================================================
# MAIN PER-RUN PREPROCESS
# ============================================================
def preprocess_one_run(run_info: Dict, out_root: Path):
    edf_path = run_info["edf_path"]
    channels_tsv = run_info["channels_tsv"]
    events_tsv = run_info["events_tsv"]
    json_path = run_info["json_path"]
    stem = run_info["stem"]
    subject = run_info["subject"]

    # subject subfolder output
    subject_out = out_root / subject
    subject_out.mkdir(parents=True, exist_ok=True)

    npz_path = subject_out / f"{stem}.npz"
    meta_path = subject_out / f"{stem}_meta.json"

    if npz_path.exists() and meta_path.exists() and not CONFIG["OVERWRITE"]:
        print(f"[SKIP] {stem}")
        return

    print(f"[RUN ] {stem}")

    ieeg_json = read_json(json_path)
    ictal_intervals = load_ictal_intervals(events_tsv)

    raw = mne.io.read_raw_edf(str(edf_path), preload=True, verbose="ERROR")
    n_channels_before_drop = len(raw.ch_names)
    sfreq_before = float(raw.info["sfreq"])

    raw = choose_ieeg_channels(raw, channels_tsv)
    n_channels_after_drop = len(raw.ch_names)

    if PIPELINE.reference == "car":
        raw.set_eeg_reference("average", projection=False)
    elif PIPELINE.reference == "bipolar":
        raw = make_bipolar_adjacent(raw)
    else:
        raise ValueError(f"Unknown reference: {PIPELINE.reference}")

    n_channels_after_reference = len(raw.ch_names)

    notch_freqs = [f for f in CONFIG["NOTCH_FREQS"] if f < raw.info["sfreq"] / 2.0]
    if notch_freqs:
        raw.notch_filter(freqs=notch_freqs, verbose="ERROR")

    raw.filter(
        l_freq=PIPELINE.band[0],
        h_freq=PIPELINE.band[1],
        verbose="ERROR",
    )

    raw.resample(PIPELINE.resample, verbose="ERROR")
    sfreq_after = float(raw.info["sfreq"])

    data = raw.get_data().astype(np.float32)

    if PIPELINE.normalize == "robust_zscore_per_channel":
        data = robust_zscore_per_channel(data).astype(np.float32)
    else:
        raise ValueError(f"Unknown normalize mode: {PIPELINE.normalize}")

    out = make_windows(
        data_ch_time=data,
        sfreq=sfreq_after,
        ictal_intervals=ictal_intervals,
        cfg=WINDOWING,
    )

    X_fixed, mask_fixed, kept_idx = pad_or_trim_channels(
        out["X"],
        target_n_channels=PIPELINE.target_n_channels,
        seed=CONFIG["RANDOM_SEED"],
    )

    mask_2d = np.broadcast_to(mask_fixed[None, :], (X_fixed.shape[0], X_fixed.shape[1]))

    save_dict = {
        "X": X_fixed.astype(np.float32),
        "y": out["y"].astype(np.int64),
        "y_train": out["y_train"].astype(np.int64),
        "t_bounds": out["t_bounds"].astype(np.float32),
        "mask": mask_2d.astype(bool),
        "rms_z": out["rms_z"].astype(np.float32),
        "peak_z": out["peak_z"].astype(np.float32),
    }

    np.savez_compressed(npz_path, **save_dict)

    # handle events_tsv path for meta
    events_tsv_str = str(events_tsv) if events_tsv is not None else ""

    meta = RunMeta(
        pipeline=asdict(PIPELINE),
        subject=run_info["subject"],
        session=run_info["session"],
        task=run_info["task"],
        acquisition=run_info["acquisition"],
        run=run_info["run"],
        edf_path=str(edf_path),
        channels_tsv=str(channels_tsv),
        events_tsv=events_tsv_str,
        json_path=str(json_path),
        recording_duration_sec=float(ieeg_json.get("RecordingDuration", len(data[0]) / sfreq_before)),
        sfreq_before_resample=sfreq_before,
        sfreq_after_resample=sfreq_after,
        powerline_frequency_hz=safe_float(ieeg_json.get("PowerLineFrequency")),
        ictal_intervals=[(float(a), float(b)) for a, b in ictal_intervals],
        windowing=asdict(WINDOWING),
        n_channels_before_drop=n_channels_before_drop,
        n_channels_after_drop=n_channels_after_drop,
        n_channels_after_reference=n_channels_after_reference,
        n_channels_after_fix128=int(X_fixed.shape[1]),
        kept_channel_indices=[int(i) for i in kept_idx.tolist()],
        channel_names_after_reference=list(raw.ch_names),
        separability={
            "rms_sep": separability_score(out["rms_z"], out["y"]),
            "peak_sep": separability_score(out["peak_z"], out["y"]),
        },
        class_counts={
            "n_total": int(len(out["y"])),
            "n_ictal": int((out["y"] == 1).sum()),
            "n_nonictal": int((out["y"] == 0).sum()),
            "n_ignore": int((out["y_train"] == -1).sum()),
            "n_train_ictal": int((out["y_train"] == 1).sum()),
            "n_train_nonictal": int((out["y_train"] == 0).sum()),
        },
    )

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(asdict(meta), f, indent=2)

    print(f"[SAVE] {npz_path}")
    print(f"[SAVE] {meta_path}")


In [7]:
# ============================================================
# MAIN PER-RUN PREPROCESS
# ============================================================
def preprocess_one_run(run_info: Dict, out_root: Path):
    edf_path = run_info["edf_path"]
    channels_tsv = run_info["channels_tsv"]
    events_tsv = run_info["events_tsv"]
    json_path = run_info["json_path"]
    stem = run_info["stem"]
    subject = run_info["subject"]

    # subject subfolder output
    subject_out = out_root / subject
    subject_out.mkdir(parents=True, exist_ok=True)

    npz_path = subject_out / f"{stem}.npz"
    meta_path = subject_out / f"{stem}_meta.json"

    if npz_path.exists() and meta_path.exists() and not CONFIG["OVERWRITE"]:
        print(f"[SKIP] {stem}")
        return

    print(f"[RUN ] {stem}")

    ieeg_json = read_json(json_path)
    ictal_intervals = load_ictal_intervals(events_tsv)

    raw = mne.io.read_raw_edf(str(edf_path), preload=True, verbose="ERROR")
    n_channels_before_drop = len(raw.ch_names)
    sfreq_before = float(raw.info["sfreq"])

    raw = choose_ieeg_channels(raw, channels_tsv)
    n_channels_after_drop = len(raw.ch_names)

    if PIPELINE.reference == "car":
        raw.set_eeg_reference("average", projection=False)
    elif PIPELINE.reference == "bipolar":
        raw = make_bipolar_adjacent(raw)
    else:
        raise ValueError(f"Unknown reference: {PIPELINE.reference}")

    n_channels_after_reference = len(raw.ch_names)

    notch_freqs = [f for f in CONFIG["NOTCH_FREQS"] if f < raw.info["sfreq"] / 2.0]
    if notch_freqs:
        raw.notch_filter(freqs=notch_freqs, verbose="ERROR")

    raw.filter(
        l_freq=PIPELINE.band[0],
        h_freq=PIPELINE.band[1],
        verbose="ERROR",
    )

    raw.resample(PIPELINE.resample, verbose="ERROR")
    sfreq_after = float(raw.info["sfreq"])

    data = raw.get_data().astype(np.float32)

    if PIPELINE.normalize == "robust_zscore_per_channel":
        data = robust_zscore_per_channel(data).astype(np.float32)
    else:
        raise ValueError(f"Unknown normalize mode: {PIPELINE.normalize}")

    out = make_windows(
        data_ch_time=data,
        sfreq=sfreq_after,
        ictal_intervals=ictal_intervals,
        cfg=WINDOWING,
    )

    X_fixed, mask_fixed, kept_idx = pad_or_trim_channels(
        out["X"],
        target_n_channels=PIPELINE.target_n_channels,
        seed=CONFIG["RANDOM_SEED"],
    )

    mask_2d = np.broadcast_to(mask_fixed[None, :], (X_fixed.shape[0], X_fixed.shape[1]))

    save_dict = {
        "X": X_fixed.astype(np.float32),
        "y": out["y"].astype(np.int64),
        "y_train": out["y_train"].astype(np.int64),
        "t_bounds": out["t_bounds"].astype(np.float32),
        "mask": mask_2d.astype(bool),
        "rms_z": out["rms_z"].astype(np.float32),
        "peak_z": out["peak_z"].astype(np.float32),
    }

    np.savez_compressed(npz_path, **save_dict)

    # handle events_tsv path for meta
    events_tsv_str = str(events_tsv) if events_tsv is not None else ""

    meta = RunMeta(
        pipeline=asdict(PIPELINE),
        subject=run_info["subject"],
        session=run_info["session"],
        task=run_info["task"],
        acquisition=run_info["acquisition"],
        run=run_info["run"],
        edf_path=str(edf_path),
        channels_tsv=str(channels_tsv),
        events_tsv=events_tsv_str,
        json_path=str(json_path),
        recording_duration_sec=float(ieeg_json.get("RecordingDuration", len(data[0]) / sfreq_before)),
        sfreq_before_resample=sfreq_before,
        sfreq_after_resample=sfreq_after,
        powerline_frequency_hz=safe_float(ieeg_json.get("PowerLineFrequency")),
        ictal_intervals=[(float(a), float(b)) for a, b in ictal_intervals],
        windowing=asdict(WINDOWING),
        n_channels_before_drop=n_channels_before_drop,
        n_channels_after_drop=n_channels_after_drop,
        n_channels_after_reference=n_channels_after_reference,
        n_channels_after_fix128=int(X_fixed.shape[1]),
        kept_channel_indices=[int(i) for i in kept_idx.tolist()],
        channel_names_after_reference=list(raw.ch_names),
        separability={
            "rms_sep": separability_score(out["rms_z"], out["y"]),
            "peak_sep": separability_score(out["peak_z"], out["y"]),
        },
        class_counts={
            "n_total": int(len(out["y"])),
            "n_ictal": int((out["y"] == 1).sum()),
            "n_nonictal": int((out["y"] == 0).sum()),
            "n_ignore": int((out["y_train"] == -1).sum()),
            "n_train_ictal": int((out["y_train"] == 1).sum()),
            "n_train_nonictal": int((out["y_train"] == 0).sum()),
        },
    )

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(asdict(meta), f, indent=2)

    print(f"[SAVE] {npz_path}")
    print(f"[SAVE] {meta_path}")


In [8]:
# ============================================================
# MAIN
# ============================================================
def main():
    raw_root: Path = CONFIG["RAW_ROOT"]
    out_root: Path = CONFIG["OUT_ROOT"]

    runs = find_runs(raw_root)

    if CONFIG["SUBJECT_FILTER"]:
        runs = [r for r in runs if r["subject"] in CONFIG["SUBJECT_FILTER"]]

    if CONFIG["TASK_FILTER"]:
        runs = [r for r in runs if r["task"] in CONFIG["TASK_FILTER"]]

    if CONFIG["ACQ_FILTER"]:
        runs = [r for r in runs if r["acquisition"] in CONFIG["ACQ_FILTER"]]

    print(f"Found {len(runs)} runs")

    for r in runs:
        try:
            preprocess_one_run(r, out_root)
        except Exception as e:
            print(f"[FAIL] {r['stem']} -> {e}")

    print("Done.")


if __name__ == "__main__":
    main()

Found 319 runs
[RUN ] sub-HUP060_ses-presurgery_task-ictal_acq-seeg_run-01
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=26, n_times=189000
    Range : 0 ... 188999 =      0.000 ...   377.998 secs
Ready.
Added the following bipolar channels:
LAF1-LAF2, LAF2-LAF3, LAF3-LAF4, RA2-RA3, RAFA1-RAFA2, RAFA2-RAFA3, RAFA3-RAFA4, RAFB1-RAFB2, RAFB2-RAFB3, RAFB3-RAFB4, RAFC1-RAFC2, RAFC2-RAFC3, RAFC3-RAFC4, RAFD1-RAFD2, RAFD2-RAFD3, RAFD3-RAFD4, RH1-RH2, RH2-RH3, RH3-RH4, RPFA1-RPFA2, RPFA2-RPFA3, RPFA3-RPFA4, RPFB1-RPFB2, RPFB2-RPFB3, RPFC1-RPFC2, RPFC2-RPFC3
[SAVE] D:\HUP_processed_ver2\sub-HUP060\sub-HUP060_ses-presurgery_task-ictal_acq-seeg_run-01.npz
[SAVE] D:\HUP_processed_ver2\sub-HUP060\sub-HUP060_ses-presurgery_task-ictal_acq-seeg_run-01_meta.json
[RUN ] sub-HUP060_ses-presurgery_task-ictal_acq-seeg_run-02
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=26, n_times=160000
    Range : 0 ... 159999

[SAVE] D:\HUP_processed_ver2\sub-HUP065\sub-HUP065_ses-presurgery_task-ictal_acq-ecog_run-02.npz
[SAVE] D:\HUP_processed_ver2\sub-HUP065\sub-HUP065_ses-presurgery_task-ictal_acq-ecog_run-02_meta.json
[RUN ] sub-HUP065_ses-presurgery_task-ictal_acq-ecog_run-03
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=63, n_times=138240
    Range : 0 ... 138239 =      0.000 ...   269.998 secs
Ready.
Added the following bipolar channels:
RG1-RG2, RG2-RG3, RG3-RG4, RG4-RG5, RG5-RG6, RG6-RG7, RG7-RG8, RG8-RG9, RG9-RG10, RG10-RG11, RG11-RG12, RG12-RG13, RG13-RG14, RG14-RG15, RG15-RG16, RG16-RG17, RG17-RG18, RG18-RG19, RG19-RG20, RG20-RG21, RG21-RG22, RG22-RG23, RG23-RG24, RG24-RG25, RG25-RG26, RG26-RG27, RG27-RG28, RG28-RG29, RG29-RG30, RG30-RG31, RG31-RG32, RG32-RG33, RG33-RG34, RG34-RG35, RG35-RG36, RG36-RG37, RG37-RG38, RG38-RG39, RG39-RG40, RG40-RG41, RG41-RG42, RG42-RG43, RG43-RG44, RG44-RG45, RG45-RG46, RG46-RG47, RG47-RG48, RG48-RG49, RG49-RG50, RG50

EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=61, n_times=95744
    Range : 0 ... 95743 =      0.000 ...   186.998 secs
Ready.
Added the following bipolar channels:
LG1-LG2, LG2-LG3, LG3-LG4, LG4-LG5, LG5-LG6, LG6-LG7, LG7-LG8, LG8-LG9, LG9-LG10, LG10-LG11, LG11-LG12, LG12-LG13, LG13-LG14, LG14-LG15, LG15-LG16, LG16-LG17, LG17-LG18, LG18-LG19, LG19-LG20, LG20-LG21, LG21-LG22, LG22-LG23, LG23-LG24, LG24-LG25, LG25-LG26, LG26-LG27, LG27-LG28, LG28-LG29, LG29-LG30, LG30-LG31, LG31-LG32, LG32-LG33, LG33-LG34, LG34-LG35, LG35-LG36, LG36-LG37, LG37-LG38, LG38-LG39, LG39-LG40, LG40-LG41, LG41-LG42, LG42-LG43, LG43-LG44, LG44-LG45, LG45-LG46, LG46-LG47, LG47-LG48, LG48-LG49, LG49-LG50, LG52-LG53, LG53-LG54, LG54-LG55, LG55-LG56, LG56-LG57, LG57-LG58, LG58-LG59, LG59-LG60, LG60-LG61, LG61-LG62, LG62-LG63, LG63-LG64
[SAVE] D:\HUP_processed_ver2\sub-HUP070\sub-HUP070_ses-presurgery_task-ictal_acq-ecog_run-05.npz
[SAVE] D:\HUP_processed_ver2\sub-HUP07

[SAVE] D:\HUP_processed_ver2\sub-HUP074\sub-HUP074_ses-presurgery_task-ictal_acq-ecog_run-03.npz
[SAVE] D:\HUP_processed_ver2\sub-HUP074\sub-HUP074_ses-presurgery_task-ictal_acq-ecog_run-03_meta.json
[RUN ] sub-HUP074_ses-presurgery_task-ictal_acq-ecog_run-04
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=103, n_times=118784
    Range : 0 ... 118783 =      0.000 ...   231.998 secs
Ready.
Added the following bipolar channels:
AD1-AD2, AD2-AD3, AD3-AD4, AD4-AD5, AD5-AD6, AD6-AD7, AD7-AD8, AIT1-AIT2, AIT2-AIT3, AIT3-AIT4, AIT4-AIT5, AIT5-AIT6, FOP1-FOP2, FOP2-FOP3, FOP3-FOP4, FOP4-FOP5, FOP5-FOP6, GRID1-GRID2, GRID2-GRID3, GRID3-GRID4, GRID4-GRID5, GRID5-GRID6, GRID6-GRID7, GRID7-GRID8, GRID8-GRID9, GRID9-GRID10, GRID10-GRID11, GRID11-GRID12, GRID12-GRID13, GRID13-GRID14, GRID14-GRID15, GRID15-GRID16, GRID16-GRID17, GRID17-GRID18, GRID18-GRID19, GRID19-GRID20, GRID20-GRID21, GRID21-GRID22, GRID22-GRID23, GRID23-GRID24, GRID24-GRID25, GRID25-GR

[SAVE] D:\HUP_processed_ver2\sub-HUP075\sub-HUP075_ses-presurgery_task-ictal_acq-ecog_run-01.npz
[SAVE] D:\HUP_processed_ver2\sub-HUP075\sub-HUP075_ses-presurgery_task-ictal_acq-ecog_run-01_meta.json
[RUN ] sub-HUP075_ses-presurgery_task-interictal_acq-ecog_run-01
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=93, n_times=153600
    Range : 0 ... 153599 =      0.000 ...   299.998 secs
Ready.
Added the following bipolar channels:
AMY1-AMY2, AMY2-AMY3, AMY3-AMY4, AMY4-AMY5, AMY5-AMY6, AMY6-AMY7, AMY7-AMY8, AST1-AST2, AST2-AST3, AST3-AST4, G1-G2, G2-G3, G5-G6, G6-G7, G7-G8, G8-G9, G9-G10, G10-G11, G11-G12, G12-G13, G13-G14, G14-G15, G17-G18, G18-G19, G19-G20, G20-G21, G21-G22, G22-G23, G23-G24, G24-G25, G25-G26, G26-G27, G27-G28, G28-G29, G29-G30, G30-G31, G31-G32, G32-G33, G33-G34, G34-G35, G35-G36, G36-G37, G37-G38, G38-G39, G39-G40, G40-G41, G41-G42, G42-G43, G43-G44, G44-G45, G45-G46, G46-G47, G49-G50, G50-G51, G51-G52, G52-G53, G53-G54, G

...

EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=100, n_times=117248
    Range : 0 ... 117247 =      0.000 ...   228.998 secs
Ready.
Added the following bipolar channels:
LA1-LA2, LA2-LA3, LA3-LA4, LA4-LA5, LA5-LA6, LA6-LA7, LA7-LA8, LA8-LA9, LA9-LA10, LB1-LB2, LB2-LB3, LB3-LB4, LB4-LB5, LB5-LB6, LB6-LB7, LB7-LB8, LB8-LB9, LB9-LB10, LB10-LB11, LB11-LB12, LC1-LC2, LC2-LC3, LC3-LC4, LC4-LC5, LC5-LC6, LC6-LC7, LC7-LC8, LC8-LC9, LC11-LC12, LD1-LD2, LD2-LD3, LD3-LD4, LD4-LD5, LD5-LD6, LD6-LD7, LD7-LD8, LD8-LD9, LD9-LD10, LD10-LD11, LD11-LD12, LE1-LE2, LE2-LE3, LE3-LE4, LE4-LE5, LE5-LE6, LE6-LE7, LE7-LE8, LE8-LE9, LE9-LE10, LE10-LE11, LE11-LE12, LF1-LF2, LF2-LF3, LF3-LF4, LF4-LF5, LF5-LF6, LF6-LF7, LF7-LF8, LF8-LF9, LF9-LF10, LF10-LF11, LG1-LG2, LG2-LG3, LG3-LG4, LG4-LG5, LG5-LG6, LG6-LG7, LG7-LG8, LG8-LG9, LG9-LG10, LG10-LG11, LG11-LG12, LH1-LH2, LH2-LH3, LH3-LH4, LH4-LH5, LH5-LH6, LH6-LH7, LH7-LH8, LH8-LH9, LH9-LH10, LH10-LH11, RA1-RA2, RA2-RA3, 

[SAVE] D:\HUP_processed_ver2\sub-HUP185\sub-HUP185_ses-presurgery_task-interictal_acq-seeg_run-01.npz
[SAVE] D:\HUP_processed_ver2\sub-HUP185\sub-HUP185_ses-presurgery_task-interictal_acq-seeg_run-01_meta.json
[RUN ] sub-HUP185_ses-presurgery_task-interictal_acq-seeg_run-02
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=100, n_times=153600
    Range : 0 ... 153599 =      0.000 ...   299.998 secs
Ready.
Added the following bipolar channels:
LA1-LA2, LA2-LA3, LA3-LA4, LA4-LA5, LA5-LA6, LA6-LA7, LA7-LA8, LA8-LA9, LA9-LA10, LB1-LB2, LB2-LB3, LB3-LB4, LB4-LB5, LB5-LB6, LB6-LB7, LB7-LB8, LB8-LB9, LB9-LB10, LB10-LB11, LB11-LB12, LC1-LC2, LC2-LC3, LC3-LC4, LC4-LC5, LC5-LC6, LC6-LC7, LC7-LC8, LC8-LC9, LC11-LC12, LD1-LD2, LD2-LD3, LD3-LD4, LD4-LD5, LD5-LD6, LD6-LD7, LD7-LD8, LD8-LD9, LD9-LD10, LD10-LD11, LD11-LD12, LE1-LE2, LE2-LE3, LE3-LE4, LE4-LE5, LE5-LE6, LE6-LE7, LE7-LE8, LE8-LE9, LE9-LE10, LE10-LE11, LE11-LE12, LF1-LF2, LF2-LF3, LF3-LF4, LF4-LF

[SAVE] D:\HUP_processed_ver2\sub-HUP187\sub-HUP187_ses-presurgery_task-interictal_acq-seeg_run-01.npz
[SAVE] D:\HUP_processed_ver2\sub-HUP187\sub-HUP187_ses-presurgery_task-interictal_acq-seeg_run-01_meta.json
[RUN ] sub-HUP187_ses-presurgery_task-interictal_acq-seeg_run-02
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=75, n_times=153600
    Range : 0 ... 153599 =      0.000 ...   299.998 secs
Ready.
Added the following bipolar channels:
RA1-RA2, RA2-RA3, RA3-RA4, RA4-RA5, RA5-RA6, RA6-RA7, RA7-RA8, RB1-RB2, RB2-RB3, RB3-RB4, RB4-RB5, RB5-RB6, RB6-RB7, RB7-RB8, RB8-RB9, RB9-RB10, RB10-RB11, RB11-RB12, RC1-RC2, RC2-RC3, RC3-RC4, RC4-RC5, RC5-RC6, RC6-RC7, RC7-RC8, RC8-RC9, RC11-RC12, RD1-RD2, RD2-RD3, RD3-RD4, RD4-RD5, RD5-RD6, RD6-RD7, RD7-RD8, RD8-RD9, RD11-RD12, RE1-RE2, RE2-RE3, RE3-RE4, RE4-RE5, RE5-RE6, RE6-RE7, RE7-RE8, RE8-RE9, RE9-RE10, RE10-RE11, RE11-RE12, RF1-RF2, RF2-RF3, RF3-RF4, RF4-RF5, RF7-RF8, RF8-RF9, RF9-RF10, RF10-RF11,

[SAVE] D:\HUP_processed_ver2\sub-HUP188\sub-HUP188_ses-presurgery_task-ictal_acq-seeg_run-04.npz
[SAVE] D:\HUP_processed_ver2\sub-HUP188\sub-HUP188_ses-presurgery_task-ictal_acq-seeg_run-04_meta.json
[RUN ] sub-HUP188_ses-presurgery_task-ictal_acq-seeg_run-05
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=145, n_times=102912
    Range : 0 ... 102911 =      0.000 ...   200.998 secs
Ready.
Added the following bipolar channels:
LA1-LA2, LA2-LA3, LA3-LA4, LA4-LA5, LA5-LA6, LA6-LA7, LA7-LA8, LA8-LA9, LA9-LA10, LB1-LB2, LB2-LB3, LB3-LB4, LB4-LB5, LB5-LB6, LB6-LB7, LB7-LB8, LB8-LB9, LB9-LB10, LB10-LB11, LB11-LB12, LC1-LC2, LC2-LC3, LC3-LC4, LC4-LC5, LC5-LC6, LC6-LC7, LC7-LC8, LC8-LC9, LC9-LC10, LC10-LC11, LC11-LC12, LD1-LD2, LD2-LD3, LD3-LD4, LD4-LD5, LD5-LD6, LD6-LD7, LD7-LD8, LD8-LD9, LD9-LD10, LD10-LD11, LD11-LD12, LE1-LE2, LE2-LE3, LE3-LE4, LE4-LE5, LE5-LE6, LE6-LE7, LE7-LE8, LE8-LE9, LE9-LE10, LE10-LE11, LE11-LE12, LF1-LF2, LF2-LF3, LF3-LF4, 

[SAVE] D:\HUP_processed_ver2\sub-HUP190\sub-HUP190_ses-presurgery_task-ictal_acq-seeg_run-02.npz
[SAVE] D:\HUP_processed_ver2\sub-HUP190\sub-HUP190_ses-presurgery_task-ictal_acq-seeg_run-02_meta.json
[RUN ] sub-HUP190_ses-presurgery_task-ictal_acq-seeg_run-03
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=132, n_times=259072
    Range : 0 ... 259071 =      0.000 ...   252.999 secs
Ready.
Added the following bipolar channels:
LA1-LA2, LA2-LA3, LA3-LA4, LA4-LA5, LA5-LA6, LA6-LA7, LA7-LA8, LA10-LA11, LA11-LA12, LB1-LB2, LB2-LB3, LB3-LB4, LB4-LB5, LB5-LB6, LB6-LB7, LB7-LB8, LB8-LB9, LB9-LB10, LC1-LC2, LC2-LC3, LC3-LC4, LC4-LC5, LC5-LC6, LC6-LC7, LC7-LC8, LC8-LC9, LC9-LC10, LC10-LC11, LC11-LC12, LD1-LD2, LD2-LD3, LD3-LD4, LD4-LD5, LD5-LD6, LD6-LD7, LD7-LD8, LD8-LD9, LD9-LD10, LD10-LD11, LD11-LD12, LG1-LG2, LG2-LG3, LG3-LG4, LG4-LG5, LG5-LG6, LG6-LG7, LG7-LG8, LG8-LG9, LG9-LG10, LG10-LG11, LG11-LG12, LH1-LH2, LH2-LH3, LH3-LH4, LH4-LH5, LH5-LH6, L